<img src="http://eikon.tpq.io/refinitiv_logo.png" width="28%" align="left" style="vertical-align: top; padding-top: 23px;">
<img src="http://hilpisch.com/tpq_logo_long.png" width="36%" align="right" style="vertical-align: top;">

# Eikon Data API

**Cross-Asset Financial Analytics &mdash; The Random Walk Hypothesis Revisited**

Dr. Yves J. Hilpisch | The Python Quants GmbH

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>

<img src="http://hilpisch.com/images/tr_eikon_02.png" width=350px align=left>

## The Agenda

This tutorial shows

* how to retrieve historical data across asset classes via the Eikon Data API,
* how to work with such data using `pandas`, `Plotly` and `Cufflinks` and
* how to derive support for the Random Walk Hypothesis from financial time series data.

## Random Walk Hypothesis

Eugene F. Fama (1965): “Random Walks in Stock Market Prices”:

> “For many years, economists, statisticians, and teachers of finance have been interested in developing and testing models of stock price behavior. One important model that has evolved from this research is the theory of random walks. This theory casts serious doubt on many other methods for describing and predicting stock price behavior—methods that have considerable popularity outside the academic world. For example, we shall see later that, if the random-walk theory is an accurate description of reality, then the various “technical” or “chartist” procedures for predicting stock prices are completely without value.”

Michael Jensen (1978): “Some Anomalous Evidence Regarding Market Efficiency”:

>“A market is efficient with respect to an information set S if it is impossible to make economic profits by trading on the basis of information set S.”

If a stock price follows a (simple) random walk (no drift & normally distributed returns), then it rises and falls with the same probability of 50% (“toss of a coin”).

**In such a case, the best predictor of tomorrow’s stock price — in a least-squares sense — is today’s stock price.**

## Importing Required Packages

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
%run credentials.ipynb
import warnings; warnings.simplefilter('ignore')

In [ ]:
import eikon as ek  # the Eikon Python wrapper package
import numpy as np  # NumPy
import pandas as pd  # pandas
import cufflinks as cf  # Cufflinks
import configparser as cp

The following **Python and package versions** are used.

In [ ]:
import sys
print(sys.version)

In [ ]:
ek.__version__

In [ ]:
np.__version__

In [ ]:
pd.__version__

In [ ]:
cf.__version__

## Connecting to Eikon Data API

This code sets the `app_key` to connect to the **Eikon Data API Proxy** .The Refinitiv workspace that runs on the desktop can act as one. It requires the app key of the refinitiv data platform

In [ ]:
# APP_KEY = 'YOUR_EIKON_APP_KEY'

In [ ]:
ek.set_app_key(APP_KEY)

## Retrieving Cross-Asset Data

We first define a **small universe of `RICS`** for which to retrieve data.

In [ ]:
rics = [
    'GE',  # General Electric stock
    'AAPL.O',  # Apple stock
    '.SPX',  # S&P 500 stock index
    '.VIX',  # VIX volatility index
    'EUR=',  # EUR/USD exchange rate
    'XAU=',  # Gold price
    'DE10YT=RR',  # 10yr Bund price
]

Second, **end-of-day (EOD) data** is retrieved.

In [ ]:
data = ek.get_timeseries(rics, fields='CLOSE',
                         start_date='2019-01-01',
                         end_date='2020-05-25')

In [ ]:
data.head()  # first five rows

In [ ]:
data.tail()  # final five rows

Only complete data rows are selected.

In [ ]:
data.dropna(inplace=True)  # deletes tows with NaN values

In [ ]:
data.info()  # DataFrame meta information

## Calculating the Log Returns

We next calculate the **log returns** in vectorized fashion.

In [ ]:
rets = np.log(data / data.shift(1))  # log returns in vectorized fashion

In [ ]:
rets.head()

`pandas` allows to derive the **correlation matrix** with a single method call.

In [ ]:
data.corr()  # correlation matrix by column

## Plotting the Data

Using `Cufflinks`, we can plot the normalized financial time series as **line plots** for comparison.

In [ ]:
cf.set_config_file(offline=True)  # set the plotting mode to offline

In [ ]:
data.normalize().iplot(kind='lines')

The frequeny distributions, i.e. the **histograms**, of the log returns per `RIC`.

In [ ]:
rets.iplot(kind='histogram', subplots=True)

The **heatmap** below visualizes the correlations between the financial time series.

In [ ]:
data.corr().iplot(kind='heatmap', colorscale='blues')

## Preparing Lagged Data

To gain insights into whether the random walk hypothesis holds true, we work with **five lags**. The code that follows derives the **lagged data** for every single `RIC`. First, a function that adds columns with lagged data to a `DataFrame` object.

In [ ]:
def add_lags(data, ric, lags):
    cols = []
    df = pd.DataFrame(data[ric])
    for lag in range(1, lags + 1):
        col = 'lag_{}'.format(lag)  # defines the column name
        df[col] = df[ric].shift(lag)  # creates the lagged data column
        cols.append(col)  # stores the column name
    df.dropna(inplace=True)  # gets rid of incomplete data rows
    return df, cols

Second, the iterations over all `RICs`, using the `add_lags` function and storing the resulting `DataFrame` objects in a dictonary.

In [ ]:
lags = 5  # five historical lags

In [ ]:
dfs = {}
for ric in rics:
    df, cols = add_lags(data, ric, lags)
    dfs[ric] = df

In [ ]:
cols  # the column names for the lags

In [ ]:
dfs.keys()  # the keys of the dictonary

In [ ]:
dfs['AAPL.O'].head(7)

## Implementing OLS Regression

The matrix consisting of the lagged data columns is used to "predict" the next day's value of the `RIC` via **linear OLS regression**.

In [ ]:
regs = {}
for ric in rics:
    df = dfs[ric]  # getting data for the RIC
    reg = np.linalg.lstsq(df[cols], df[ric], rcond=-1)[0]  # the OLS regression
    regs[ric] = reg  # storing the results

In [ ]:
for ric in rics:
    print('{:10} | {}'.format(ric, regs[ric]))

## Taking a Closer Look

Let's pick one `RIC` and compare the original time series with the OLS predicted one.

In [ ]:
ric = 'AAPL.O'

In [ ]:
res = pd.DataFrame(dfs[ric][ric])  # picks the original time series

In [ ]:
res['PRED'] = np.dot(dfs[ric][cols], regs[ric])  # creates the "prediction" values

The **predicted prices** are almost exactly the prices from the day before.

In [ ]:
res.iloc[-50:].iplot()

In [ ]:
res.head()

## Analyzing the Results

Now analyzing the **regression results** a bit more formally.

In [ ]:
rega = np.stack(regs.values())  # combines the regression results
rega

Almost all the weight lies on the most recent price (`lag_1`).

In [ ]:
rega.mean(axis=0)  # mean values by column

In [ ]:
regd = pd.DataFrame(rega, columns=cols, index=rics)  # converting the results to DataFrame

In [ ]:
regd

In [ ]:
regd.describe()  # summary statistics

## Visualizing the Results

The following bar chart illustrates that the results a qualitatively similar for all `RICS` analyzed &mdash; "_today's price is the best predictor, in a least-squares sense, for tomorrow's price_".

In [ ]:
regd.iplot(kind='bar')

The **mean values** for the single optimal regression parameters.

In [ ]:
regd.mean().iplot(kind='bar')

## Analyzing Intraday Data

Let us quickly check, whether the results are similar on an **intraday basis**.

In [ ]:
data = ek.get_timeseries(rics,  # RICs
              fields='CLOSE',  # fields to be retrieved
              start_date='2020-05-08 14:00:00',  # start time
              end_date='2020-05-08 18:00:00',  # end time
              interval='minute')  # bar length

In [ ]:
data.info()

In [ ]:
data.tail()

In [ ]:
dfs = {}
for ric in rics:
    df, cols = add_lags(data, ric, lags)
    dfs[ric] = df

In [ ]:
regs = {}
for ric in rics:
    df = dfs[ric]
    reg = np.linalg.lstsq(df[cols], df[ric], rcond=-1)[0]
    regs[ric] = reg

In [ ]:
rega = np.stack(regs.values())

In [ ]:
regd = pd.DataFrame(rega, columns=cols, index=rics)

**Intraday** the optimal regression parameters show more variation.

In [ ]:
regd.iplot(kind='bar')

In [ ]:
regd.mean().iplot(kind='bar')

## Conclusions

Based on this tutorial, we can conclude that

* it is easy to retrieve **historical end-of-day and intraday data across asset classes** via the Eikon Data API,
* `Plotly` and `Cufflinks` make **financial data visualization** convenient and
* there is **support for the Random Walk Hypothesis** based on the OLS regression analysis (both daily and a little bit less so intraday).

## Eikon Data API Developer Resources

* [Overview](https://developers.thomsonreuters.com/eikon-data-apis) 
* [Quick Start ](https://developers.thomsonreuters.com/eikon-data-apis/quick-start)
* [Documentation](https://developers.thomsonreuters.com/eikon-data-apis/docs)
* [Downloads](https://developers.thomsonreuters.com/eikon-data-apis/downloads)
* [Tutorials](https://developers.thomsonreuters.com/eikon-data-apis/learning)
* [Q&A Forums](https://developers.thomsonreuters.com/eikon-data-apis/qa) 

Data Item Browser Application: Type `DIB` into Eikon Search Bar.

<img src="http://eikon.tpq.io/refinitiv_logo.png" width="28%" align="left" style="vertical-align: top; padding-top: 23px;">
<img src="http://hilpisch.com/tpq_logo_long.png" width="36%" align="right" style="vertical-align: top;">